In [1]:
print(123)

123


In [2]:
import sys
print(sys.executable)

/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/.venv/bin/python


In [3]:
query = "I just discovered the course, can I still join?"

In [4]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
import sys
import numpy as np
from sentence_transformers import SentenceTransformer

print(sys.executable)
print(np.__version__)
print("OK")

/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh/.venv/bin/python
1.26.4
OK


#1. Query to turn in vektor
#2. then in vektorsearch 

In [ ]:
import sys
print(sys.executable)

import numpy
import torch
import transformers
import sentence_transformers

print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh/.venv/bin/python
numpy: 1.26.4
torch: 2.2.2
transformers: 4.41.2
sentence-transformers: 2.7.0


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
import numpy as np
import numpy as np

print(np.__version__)

1.26.4


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import sys
import numpy as np
import torch

print(sys.executable)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh/.venv/bin/python
numpy: 1.26.4
torch: 2.2.2


In [ ]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [ ]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
v1.dot(dv)

0.323324

In [ ]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [ ]:
v2.dot(dv)

0.019730484

### Embedding


In [ ]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [ ]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [ ]:
score =[]
for i in range(len(vectors)):
    score.append(v1.dot(vectors[i]))    
    score.append(score)


In [ ]:
#turn inbig matrix
import numpy as np
X = np.array(vectors)

In [ ]:
print(X)

[[-0.02670623 -0.12245761  0.01594418 ... -0.00230648 -0.11218392
  -0.02365561]
 [-0.01099556 -0.11074749 -0.02536943 ...  0.09022231 -0.02697358
   0.01975667]
 [-0.08896551 -0.06128191  0.00775606 ...  0.04059704  0.00479287
  -0.02745943]
 ...
 [-0.03652926  0.0141543  -0.06838646 ...  0.04316786  0.08105535
  -0.0214863 ]
 [-0.13091592 -0.06990603 -0.00931881 ... -0.00044338 -0.01285732
   0.0142692 ]
 [-0.07984783  0.01926981  0.02544976 ... -0.03368022 -0.01884021
   0.05837053]]


In [ ]:
# we do vektor multiplikation and very fast, because we use numpy and matrix multiplication
scores =X.dot(v1)

In [ ]:
X.shape

(1350, 384)

In [ ]:
scores

array([ 0.48740584,  0.20991942,  0.76294124, ..., -0.08637965,
        0.03759789, -0.03037038], dtype=float32)

In [ ]:
import numpy as np

In [ ]:
idx = np.argmax(scores)
idx, scores[idx]

(2, 0.76294124)

In [ ]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [ ]:
top5 = np.argsort(scores)[-5:]

In [ ]:
top5

array([  7, 538, 907, 625,   2])

In [ ]:
scores[top5]

array([0.5601001 , 0.6536312 , 0.71921325, 0.7579372 , 0.76294124],
      dtype=float32)

In [ ]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.5601001
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.', 'doc_id': '068529125b'}

0.6536312
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}

0.71921325
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can re

In [ ]:
#the judge rulled out the possibility of a crime
#let's use llm as a judge approach to evaluate our llm

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
vindex.search(v1)

[{'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'doc_id': '3f1424af17'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.",
  'doc_id': '2d8b16c2a0'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'quest

In [ ]:
vindex.search(v1, num_results=5)

[{'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'doc_id': '3f1424af17'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.",
  'doc_id': '2d8b16c2a0'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'quest

In [ ]:
vindex.search(v1, num_results=5, filter_dict={"course": "llm-zoomcamp"})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

## 6.Rag with vector search

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
from dotenv import dotenv_values, load_dotenv
from openai import OpenAI
from pathlib import Path
import os

project_root = Path("/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh")
env_path = project_root / ".env"

load_dotenv(env_path, override=True)

api_key = os.environ.get("GROQ_API_KEY") or dotenv_values(env_path).get("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY fehlt. Trage ihn in die .env-Datei ein oder setze ihn vor dem Kernel-Start als Umgebungsvariable."
    )

llm_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key,
)


In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
)


In [ ]:
import importlib
import rag_helper

importlib.reload(rag_helper)

from rag_helper import RAGBase

In [ ]:
assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    model="llama-3.3-70b-versatile",
)


In [ ]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still sign up for the program. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted. Additionally, while homework is not mandatory, it is recommended to reinforce concepts and can impact your rank on the leaderboard. The key requirement for getting a certificate is passing the Capstone project.'

In [ ]:
assistant.search(query)

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.',
  'doc_id': '9f689c185f'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'Do I need a new GitHub repo for Module 2, or just a new codespace?',
  'answer': "Just a new codespace. A codespace is an environment (see *Can I run the course locally instead of Codespaces?*); you creat

In [ ]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=llm_client,
    model="llama-3.3-70b-versatile",
)


NameError: name 'RAGVector' is not defined

In [ ]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up for the course even though it has already begun. According to the provided context, registration is not strictly necessary to participate, and you can start learning and submitting homework as long as the submission form is open. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.'

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [ ]:
vs_index.fit(vectors, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [ ]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

### Filtering

In [ ]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

### reopening the index

In [ ]:
vs_index.close()

### Using sqlitesearch vector search in RAG
